In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI
import os
from dotenv import load_dotenv
load_dotenv()

api_key = "YOUR_GOOGLE_API_KEY"  # Replace with your actual API key or use environment variable

# Load Gemini model
llm = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash",
    google_api_key=api_key,
    temperature=0.7
)                    

In [ ]:
from langchain_community.utilities import SQLDatabase

db_user = "root"
db_password = "YourPassword"  # Replace with your actual password or use environment variable
db_host = "localhost"
db_name = "retail_shop"

db = SQLDatabase.from_uri(f"mysql+pymysql://{db_user}:{db_password}@{db_host}/{db_name}" , sample_rows_in_table_info=3)
print(db.table_info)


CREATE TABLE discounts (
	discount_id INTEGER NOT NULL AUTO_INCREMENT, 
	t_shirt_id INTEGER NOT NULL, 
	pct_discount DECIMAL(5, 2), 
	PRIMARY KEY (discount_id), 
	CONSTRAINT discounts_ibfk_1 FOREIGN KEY(t_shirt_id) REFERENCES t_shirts (t_shirt_id), 
	CONSTRAINT discounts_chk_1 CHECK ((`pct_discount` between 0 and 100))
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from discounts table:
discount_id	t_shirt_id	pct_discount
1	1	10.00
2	2	15.00
3	3	20.00
*/


CREATE TABLE t_shirts (
	t_shirt_id INTEGER NOT NULL AUTO_INCREMENT, 
	brand ENUM('Van Huesen','Levi','Nike','Adidas') NOT NULL, 
	color ENUM('Red','Blue','Black','White') NOT NULL, 
	size ENUM('XS','S','M','L','XL') NOT NULL, 
	price INTEGER, 
	stock_quantity INTEGER NOT NULL, 
	PRIMARY KEY (t_shirt_id), 
	CONSTRAINT t_shirts_chk_1 CHECK ((`price` between 10 and 50))
)ENGINE=InnoDB COLLATE utf8mb4_0900_ai_ci DEFAULT CHARSET=utf8mb4

/*
3 rows from t_shirts table:
t_shirt_id	brand	color	size	price	stock

In [4]:
from langchain_experimental.sql import SQLDatabaseChain
db_chain = SQLDatabaseChain.from_llm(llm ,db , verbose =True)
response = db_chain.run("How many t-shirts do we have  left for nike in extra small size and white color?")
response


C:\Users\rajiv\AppData\Local\Temp\ipykernel_18488\2064797223.py:1: DeprecationWarning: `langchain-experimental` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-experimental/issues/87 for details.
  from langchain_experimental.sql import SQLDatabaseChain
C:\Users\rajiv\AppData\Local\Temp\ipykernel_18488\2064797223.py:3: LangChainDeprecationWarning: The method `Chain.run` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  response = db_chain.run("How many t-shirts do we have  left for nike in extra small size and white color?")




> Entering new SQLDatabaseChain chain...
How many t-shirts do we have  left for nike in extra small size and white color?
SQLQuery:SQLQuery: SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White' LIMIT 5
SQLResult: [(23,)]
Answer:Question: How many t-shirts do we have left for nike in extra small size and white color?
SQLQuery:SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White' LIMIT 5
> Finished chain.


"Question: How many t-shirts do we have left for nike in extra small size and white color?\nSQLQuery:SELECT `stock_quantity` FROM `t_shirts` WHERE `brand` = 'Nike' AND `size` = 'XS' AND `color` = 'White' LIMIT 5"

In [10]:
qns2 = db_chain.run("How much is the price of the inventory for all small size t-shirts?")
qns2



> Entering new SQLDatabaseChain chain...
How much is the price of the inventory for all small size t-shirts?
SQLQuery:SQLQuery: SELECT SUM(`price` * `stock_quantity`) AS `total_inventory_price` FROM `t_shirts` WHERE `size` = 'S';
SQLResult: [(Decimal('25373'),)]
Answer:Answer: The total price of the inventory for all small size t-shirts is 25373.
> Finished chain.


'Answer: The total price of the inventory for all small size t-shirts is 25373.'

In [11]:
qns3 = db_chain.run("If we have to sell all the Levi's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?")
qns3



> Entering new SQLDatabaseChain chain...
If we have to sell all the Levi's T-shirts today with discounts applied. How much revenue our store will generate (post discounts)?
SQLQuery:SQLQuery: SELECT SUM(T1.`stock_quantity` * T1.`price` * (1 - T2.`pct_discount` / 100)) FROM `t_shirts` AS T1 INNER JOIN `discounts` AS T2 ON T1.`t_shirt_id` = T2.`t_shirt_id` WHERE T1.`brand` = 'Levi'
SQLResult: [(Decimal('3097.400000'),)]
Answer:Our store will generate 3097.40 in revenue if all Levi's T-shirts are sold today with discounts applied.
> Finished chain.


"Our store will generate 3097.40 in revenue if all Levi's T-shirts are sold today with discounts applied."

In [12]:
qns4 =db_chain.run("Which products are comfortable for summer workouts?")
qns4



> Entering new SQLDatabaseChain chain...
Which products are comfortable for summer workouts?
SQLQuery:I am sorry, but I cannot answer the question "Which products are comfortable for summer workouts?" because the provided tables do not contain information about product comfort, suitability for summer, or workout specific attributes.

ProgrammingError: (pymysql.err.ProgrammingError) (1064, 'You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near \'I am sorry, but I cannot answer the question "Which products are comfortable for\' at line 1')
[SQL: I am sorry, but I cannot answer the question "Which products are comfortable for summer workouts?" because the provided tables do not contain information about product comfort, suitability for summer, or workout specific attributes.]
(Background on this error at: https://sqlalche.me/e/20/f405)

In [5]:
products = [
    "Nike white XS t-shirt made for lightweight summer workouts",
    "Nike black gym t-shirt with breathable fabric",
    "Adidas running t-shirt suitable for outdoor exercise",
    "Levi casual cotton t-shirt for daily comfort",
    "Van Huesen premium office wear t-shirt",
    "Nike stretchable sports wear for training sessions",
    "Adidas moisture control fitness t-shirt",
    "Levi soft cotton relaxed fit clothing",
    "Van Huesen formal casual wear for professionals"
]

In [6]:
from langchain_community.vectorstores import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# embedding model
embedding_function = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

# create chroma db
vectordb = Chroma.from_texts(
    texts=products,
    embedding=embedding_function,
    persist_directory="./chroma_db"
)

C:\Users\rajiv\AppData\Local\Temp\ipykernel_18488\3845608917.py:5: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the `langchain-huggingface package and should be used instead. To use it run `pip install -U `langchain-huggingface` and import as `from `langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_function = HuggingFaceEmbeddings(


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [7]:
def rag_search(query):

    docs = vectordb.similarity_search(query, k=2)

    context = "\n".join([doc.page_content for doc in docs])

    prompt = f"""
    Use the context below to answer the question.

    Context:
    {context}

    Question:
    {query}
    """

    response = llm.invoke(prompt)

    return response.content

In [8]:
def llm_router(question):

    router_prompt = f"""
    You are an AI router.

    Decide whether the user question should use:

    1. SQL
       - inventory
       - stock
       - quantities
       - prices
       - structured business data

    OR

    2. RAG
       - recommendations
       - semantic meaning
       - product descriptions
       - general understanding

    Return ONLY one word:
    SQL
    or
    RAG

    Question:
    {question}
    """

    route = llm.invoke(router_prompt).content.strip()

    print("Route Chosen:", route)

    # SQL route
    if route.upper() == "SQL":

        return db_chain.run(question)

    # RAG route
    else:

        return rag_search(question)

In [17]:
print(llm_router("which clothes are better for summer days"))

Route Chosen: RAG
Both the Nike breathable running t-shirt and the Adidas lightweight gym wear are suitable for summer days, as they are described as being designed for hot weather and workouts.


In [18]:
print(llm_router("which should i buy for winter season"))

Route Chosen: RAG
You should buy the **Levi heavy cotton winter hoodie** for the winter season.


In [9]:
print(llm_router("and how many winter hoodie i have of levi in my inventory"))

Route Chosen: SQL


> Entering new SQLDatabaseChain chain...
and how many winter hoodie i have of levi in my inventory
SQLQuery:I am sorry, but I cannot answer this question. The table `t_shirts` does not contain information about "winter hoodies" or any specific type of clothing beyond t-shirts.

ProgrammingError: (pymysql.err.ProgrammingError) (1064, "You have an error in your SQL syntax; check the manual that corresponds to your MySQL server version for the right syntax to use near 'I am sorry, but I cannot answer this question. The table `t_shirts` does not con' at line 1")
[SQL: I am sorry, but I cannot answer this question. The table `t_shirts` does not contain information about "winter hoodies" or any specific type of clothing beyond t-shirts.]
(Background on this error at: https://sqlalche.me/e/20/f405)